<a href="https://colab.research.google.com/github/shilpitha-03/VideoRAG/blob/surgery/surgical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NOTES:

Do not intiliaze CUDA before indexing.

In [1]:
from google.colab import drive

drive.mount('/content/surgical_drive')
print("Drive mounted on /content/surgical_drive/MyDrive")

Mounted at /content/surgical_drive
Drive mounted on /content/surgical_drive/MyDrive


In [2]:
!nvidia-smi #check we are using a100
!df -h /content/surgical_drive/MyDrive #check space on drive

Sun May 31 16:43:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
#SETUP PATHS
import os

if not os.path.exists("/content/surgical_drive/MyDrive"):
    raise RuntimeError("Drive isn't mounted!")
print("Drive mounted.")

run_id   = "001"
run_note = "baseline"

drive_root = '/content/surgical_drive/MyDrive/videorag_surgical'
run_dir    = f"{drive_root}/runs/{run_id}"

drive_paths = {
    "input_videos": f"{drive_root}/input_videos",   # same across all runs
    "workdir":      f"{run_dir}/workdir",            # index, per run
    "precompute":   f"{drive_root}/precompute",      # shared stable artifacts (later)
    "inspection":   f"{run_dir}/inspection_outputs", # fixed typo: was inspectio_outputs
}

disk_paths = {
    "weights":   "/content/model_weights",
    "whisper":   "/content/model_weights/faster-distil-whisper-large-v3",
    "minicpm":   "/content/model_weights/MiniCPM-V-2_6",
    "imagebind": "/content/model_weights/imagebind_huge.pth",
    "cache":     "/content/videorag_cache",
}

# Create folders FIRST
for name, path in drive_paths.items():
    os.makedirs(path, exist_ok=True)
    print(f"{name}: {path}")

for name, path in disk_paths.items():
    if path.endswith('.pth'):
        continue
    os.makedirs(path, exist_ok=True)
    print(f"{name}: {path}")

# THEN write the run note (run_dir now exists)
with open(f"{run_dir}/run_note.txt", 'w') as f:
    f.write(run_note)
print(f"\n✓ Run note written to {run_dir}/run_note.txt")

print("All paths set.")

Drive mounted.
input_videos: /content/surgical_drive/MyDrive/videorag_surgical/input_videos
workdir: /content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir
precompute: /content/surgical_drive/MyDrive/videorag_surgical/precompute
inspection: /content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs
weights: /content/model_weights
whisper: /content/model_weights/faster-distil-whisper-large-v3
minicpm: /content/model_weights/MiniCPM-V-2_6
cache: /content/videorag_cache

✓ Run note written to /content/surgical_drive/MyDrive/videorag_surgical/runs/001/run_note.txt
All paths set.


In [ ]:
# !pip install -U yt-dlp -q

In [ ]:
# !curl -fsSL https://deno.land/install.sh | sh
# import os

# os.environ['PATH'] += ":/root/.deno/bin"
# !/root/.deno/bin/deno --version

In [ ]:
# from google.colab import files
# files.upload()

In [ ]:
# # ── ONE-TIME VIDEO DOWNLOAD ───────────────────────────────────────────────
# # Run this MANUALLY, once, the first session only.
# # Downloads the surgical playlist videos to Drive (persistent).
# # After this has run successfully once, never run it again — Cell 8 loads
# # from Drive on every subsequent session.

# PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLSMkFPIqArg5XIoUT6-cLHHepRPaYNMBt"
# MAX_VIDEOS   = 6        # first 6 of the 14-video playlist

# import subprocess, json, re, os

# raw = subprocess.run(
#     ["yt-dlp", "--flat-playlist", "--dump-json", PLAYLIST_URL],
#     capture_output=True, text=True
# ).stdout.strip()
# entries = [json.loads(l) for l in raw.split("\n") if l][:MAX_VIDEOS]

# def slug(title):
#     return re.sub(r'[^a-zA-Z0-9]+', '_', title.lower()).strip('_')[:40]

# videos = {f"surg_{i:02d}_{slug(e.get('title','untitled'))}": e['url']
#           for i, e in enumerate(entries)}

# os.makedirs(drive_paths['input_videos'], exist_ok=True)

# for name, url in videos.items():
#     final_path = f"{drive_paths['input_videos']}/{name}.mp4"
#     if os.path.exists(final_path):
#         print(f"✓ {name} already on Drive, skipping")
#         continue
#     print(f"Downloading {name}...")
#     !yt-dlp -o "{final_path}" \
#             --format "bestvideo[ext=mp4]+bestaudio/best[ext=mp4]/best" \
#             --cookies /content/www.youtube.com_cookies.txt \
#             --extractor-args "youtube:player_client=default" \
#             --remote-components ejs:github \
#             --merge-output-format mp4 "{url}"

# print("\n=== Download complete. Videos on Drive: ===")
# for f in sorted(os.listdir(drive_paths['input_videos'])):
#     print(f"  {f}")

In [ ]:
# # Two replacements for the age-gated surg_03 and surg_04
# replacements = {
#     "surg_03_replacement": "R3i1P6_ARvI",
#     "surg_04_replacement": "48LD2S6T0uI",
# }

# for name, vid in replacements.items():
#     final_path = f"{drive_paths['input_videos']}/{name}.mp4"
#     if os.path.exists(final_path):
#         print(f"✓ {name} already on Drive, skipping"); continue
#     print(f"Downloading {name}...")
#     !yt-dlp -o "{final_path}" \
#             --format "bestvideo[ext=mp4]+bestaudio/best[ext=mp4]/best" \
#             --remote-components ejs:github \
#             --merge-output-format mp4 "https://www.youtube.com/watch?v={vid}"
#     if os.path.exists(final_path):
#         print(f"✓ {name}: {os.path.getsize(final_path)/1e6:.0f}MB\n")
#     else:
#         print(f"✗ {name}: failed (try another candidate)\n")

In [ ]:
# import glob, subprocess, os

# for p in sorted(glob.glob(f"{drive_paths['input_videos']}/*.mp4")):
#     # ffprobe reads the container and reports duration; fails loudly if corrupt
#     out = subprocess.run(
#         ["ffprobe", "-v", "error", "-show_entries", "format=duration",
#          "-of", "default=noprint_wrappers=1:nokey=1", p],
#         capture_output=True, text=True
#     )
#     dur = out.stdout.strip()
#     size = os.path.getsize(p) / 1e6
#     status = f"{float(dur)/60:.1f} min" if dur else f"✗ UNREADABLE: {out.stderr[:80]}"
#     print(f"{os.path.basename(p):55} {size:6.0f}MB  {status}")

surg_00_4k_uncinectomy_and_middle_meatal_antrost.mp4        81MB  4.5 min
surg_01_4k_anterior_ethmoidectomy_prof_simon_car.mp4        96MB  2.1 min
surg_02_4k_frontal_recess_dissection_intact_bull.mp4       113MB  5.2 min
surg_03_replacement.mp4                                    152MB  7.7 min
surg_04_replacement.mp4                                    115MB  6.3 min
surg_05_basic_fess_step_by_step.mp4                         46MB  14.9 min


In [12]:
#CLONE REPO + CHECKOUT surgery
import os

if not os.path.exists("/content/VideoRAG"):
    !git clone https://github.com/shilpitha-03/VideoRAG.git /content/VideoRAG

%cd /content/VideoRAG
!git fetch origin
!git checkout surgery
!git pull origin surgery

%cd /content/VideoRAG/VideoRAG-algorithm
!git branch
print("--- fix check (both must be > 0) ---")
!grep -c "caption_in_main_process" videorag/videorag.py
!grep -ic "bge" videorag/_llm.py

/content/VideoRAG
Already on 'surgery'
Your branch is up to date with 'origin/surgery'.
From https://github.com/shilpitha-03/VideoRAG
 * branch            surgery    -> FETCH_HEAD
Already up to date.
/content/VideoRAG/VideoRAG-algorithm
  main
* surgery
--- fix check (both must be > 0) ---
2
26


!cd changes directory in the subprocess so here this cell, but %cd is persistent across the whole session.

In [5]:
#INSTALL DEPENDENCIES
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
import torch
print(f"PyTorch: {torch.__version__}")

# dont call torch.cuda.is_available() here as it initializes cuda and we dont want to do that before indexing, because the multiprocessing fork in indexing cell recieves a broken context and crashes. so to check gpu use nvidia-smi, a subprocess that doesnt touch pytorch

PyTorch: 2.11.0+cu128


In [6]:
!pip install accelerate bitsandbytes -q
!pip install moviepy==1.0.3 -q
!pip install timm ftfy regex einops fvcore eva-decord==0.6.1 iopath -q
!pip install ctranslate2==4.4.0 faster_whisper==1.0.3 -q
!pip install hnswlib xxhash nano-vectordb neo4j -q
!pip install transformers==4.43.3 -q
!pip install tiktoken openai tenacity -q
!pip install yt-dlp -q
!pip install ollama==0.5.3 -q

!pip install --no-deps \
    git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d -q
!pip install --no-deps \
    git+https://github.com/facebookresearch/ImageBind.git@3fcf5c9039de97f6ff5528ee4a9dce903c5979b3 -q

import importlib.util
print("\n=== install check ===")
for pkg in ["imagebind", "pytorchvideo", "faster_whisper", "transformers", "moviepy"]:
    print(f"  {'✓' if importlib.util.find_spec(pkg) else '✗ MISSING'} {pkg}")

#transformers 4.37.1 in readme lacks all_tied_weights_keys that minicpmv needs and 4.5.0 breaks cause of peft/sentence-transformers 4.43.3 is the best
#ctanslate2 and faster_whisper are compiled against and incompatible cuDNN that is fixed later through symlink
#pytorchvideo and imagebnd are installed from commit hashes without dependencies, because we dont want pip to fallback to other pytorch versions which we already resolved for this setup

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 103.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 9.1 MB/s et

In [7]:
import importlib, transformers
importlib.reload(transformers)
print(transformers.__version__)
#we pip installed transformers after python already imported it earlier in the session, the old version is cached in memory, importlib.reload forces python to re-read the freshly-installed version from disk and the print verifies which version the notebook is now lookign at.

4.43.3


In [8]:
#from pytorchvideo functional_tensor module was used, but this mdule isnt available with newer torchvision 0.17+ which ships with torch 2.2+, and we cant downgrade torchvision that breaks blackwell/colab, so fake the missing module
import torchvision.transforms.functional as F
import sys, types

fake_module = types.ModuleType('torchvision.transforms.functional_tensor')
for func_name in ['rgb_to_grayscale','adjust_brightness','adjust_contrast','adjust_saturation','adjust_hue']:
  if hasattr(F, func_name):
    setattr(fake_module, func_name, getattr(F, func_name))

sys.modules['torchvision.transforms.functional_tensor']=fake_module

What this does, conceptually: the functions pytorchvideo wants still exist in torchvision — they just moved from functional_tensor to functional. So you:

types.ModuleType(...) — construct an empty module object out of thin air, named exactly what pytorchvideo will look for.
Loop over the specific functions pytorchvideo imports, and copy each from the real new location (F) onto your fake module via setattr/getattr (these are the reflection functions: getattr(F, "adjust_hue") fetches the function by string name; setattr(fake, "adjust_hue", ...) attaches it).
sys.modules['torchvision.transforms.functional_tensor'] = fake_module — this is the magic line. sys.modules is Python's live registry of imported modules. By inserting your fake here before pytorchvideo imports, when pytorchvideo runs from torchvision.transforms.functional_tensor import adjust_hue, Python finds your shim already in the registry and uses it — never noticing the real module is gone.

It's a redirect: "when anyone asks for the deleted module, hand them this stand-in that forwards to the new location." The try: from pytorchvideo.transforms import augmix at the end verifies the shim worked by doing the exact import that used to fail.

In [9]:
import subprocess
import os

# ctranslate2 bundles cuDNN 8 internally; bitsandbytes (and the linker) look for
# libcudnn_ops_infer.so.8 etc. in the system path. Symlink ctranslate2's bundled
# copy to where they're expected so the .so.8 lookups resolve.
cudnn8_source = '/usr/local/lib/python3.12/dist-packages/ctranslate2.libs/libcudnn-463fd6d5.so.8.9.7'

symlink_targets = [
    '/usr/lib/x86_64-linux-gnu/libcudnn_ops_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_cnn_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_cnn_train.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_ops_train.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_adv_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_adv_train.so.8',
]

for target in symlink_targets:
    if not os.path.exists(target):
        result = subprocess.run(
            ['ln', '-sf', cudnn8_source, target],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✓ Created: {os.path.basename(target)}")
        else:
            print(f"✗ Failed: {target} — {result.stderr}")
    else:
        print(f"✓ Already exists: {os.path.basename(target)}")

subprocess.run(['ldconfig'], capture_output=True)
print("\n✓ Library cache updated — cuDNN 8 now resolvable")

✓ Created: libcudnn_ops_infer.so.8
✓ Created: libcudnn_cnn_infer.so.8
✓ Created: libcudnn_cnn_train.so.8
✓ Created: libcudnn_ops_train.so.8
✓ Created: libcudnn_adv_infer.so.8
✓ Created: libcudnn_adv_train.so.8

✓ Library cache updated — cuDNN 8 now resolvable


The failure chain:

Colab's system has cuDNN 9 installed globally.
Whisper's engine (CTranslate2) was compiled against cuDNN 8 and bundles its own copy inside its package folder.
bitsandbytes (and other libs) look for cuDNN 8 files (libcudnn_ops_infer.so.8, etc.) in the standard system location /usr/lib/x86_64-linux-gnu/ — don't find them (only cuDNN 9 is there) — and crash on load.

The fix: create symlinks (symbolic links — filesystem pointers) from the system locations where libs look for cuDNN 8, to the actual cuDNN 8 file bundled inside CTranslate2:

ln -sf source target is the shell command to make a symlink: "create a file at target that's really a pointer to source." -s = symbolic, -f = force (overwrite if exists).
It loops over every cuDNN 8 .so.8 name that gets requested, pointing each at CTranslate2's bundled copy.
ldconfig rebuilds the system's library cache so the dynamic linker actually sees the new symlinks.

Net effect: any library that asks the system for "cuDNN 8" gets transparently routed to the copy CTranslate2 already shipped. No version conflict.

In [10]:
!pip install -q huggingface_hub

In [11]:
import os
import subprocess

# ── WHISPER ───────────────────────────────────────────────────────────────
# Check for actual weight file, not just directory existence
# 4.0K directory = empty = git lfs didn't download weights
whisper_model_bin = f"{disk_paths['whisper']}/model.bin"
whisper_ok = os.path.exists(whisper_model_bin) and os.path.getsize(whisper_model_bin) > 1e9

if not whisper_ok:
    print("Downloading Whisper (~1.5GB)...")
    if os.path.exists(disk_paths['whisper']):
        import shutil
        shutil.rmtree(disk_paths['whisper'])
    !git lfs install
    !git clone https://huggingface.co/Systran/faster-distil-whisper-large-v3 \
        {disk_paths['whisper']}
else:
    print("✓ Whisper already on local disk with real weights")

# Verify
size_gb = os.path.getsize(whisper_model_bin) / 1e9
print(f"  model.bin: {size_gb:.2f} GB {'✓' if size_gb > 1 else '✗ pointer file!'}")

# ── MINICPM-V ─────────────────────────────────────────────────────────────

from huggingface_hub import login, snapshot_download
import os
import shutil

# LOGIN FIRST
# Paste your HF token when prompted
login()

# Expected shard files
minicpm_shard1 = f"{disk_paths['minicpm']}/model-00001-of-00004.safetensors"
minicpm_shard4 = f"{disk_paths['minicpm']}/model-00004-of-00004.safetensors"
# previously assumed 8 shards, but its actually 4 shards only.
# Verify existing download
minicpm_ok = (
    os.path.exists(minicpm_shard1) and
    os.path.getsize(minicpm_shard1) > 1e9 and
    os.path.exists(minicpm_shard4) and
    os.path.getsize(minicpm_shard4) > 1e9
)

if not minicpm_ok:

    print("\nDownloading MiniCPM-V full model (~16GB, one-time)...")

    # Remove broken partial download
    if os.path.exists(disk_paths['minicpm']):
        shutil.rmtree(disk_paths['minicpm'])

    # Download model
    snapshot_download(
        repo_id="openbmb/MiniCPM-V-2_6",
        local_dir=disk_paths['minicpm'],
        local_dir_use_symlinks=False,
        resume_download=True
    )

else:
    print("\n✓ MiniCPM-V already on local disk with real weights")

# Verify shards
for shard in [
    'model-00001-of-00004.safetensors',
    'model-00004-of-00004.safetensors'
]:
    shard_path = f"{disk_paths['minicpm']}/{shard}"

    if os.path.exists(shard_path):
        size_gb = os.path.getsize(shard_path) / 1e9

        print(
            f"  {shard}: {size_gb:.2f} GB "
            f"{'✓' if size_gb > 1 else '✗ pointer/small file!'}"
        )
    else:
        print(f"  {shard}: ✗ NOT FOUND")

# ── IMAGEBIND ─────────────────────────────────────────────────────────────
# ImageBind is a single .pth file downloaded via wget - reliable
imagebind_ok = os.path.exists(disk_paths['imagebind']) and \
               os.path.getsize(disk_paths['imagebind']) > 1e9

if not imagebind_ok:
    print("\nDownloading ImageBind (~2GB)...")
    !wget -q https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth \
        -O {disk_paths['imagebind']}
else:
    print("\n✓ ImageBind already on local disk")

size_gb = os.path.getsize(disk_paths['imagebind']) / 1e9
print(f"  imagebind_huge.pth: {size_gb:.2f} GB {'✓' if size_gb > 1 else '✗ too small!'}")

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────
print("\n=== Model verification summary ===")
checks = [
    ("Whisper",    whisper_model_bin,  1e9),
    ("ImageBind",  disk_paths['imagebind'], 1e9),
    ("MiniCPM shard1", minicpm_shard1, 1e9),
]
all_ok = True
for name, path, min_size in checks:
    ok = os.path.exists(path) and os.path.getsize(path) > min_size
    print(f"  {'✓' if ok else '✗'} {name}")
    if not ok:
        all_ok = False

if all_ok:
    print("\n✓ All models ready. Safe to continue.")
else:
    raise RuntimeError("✗ Some models missing or incomplete - check output above")

Updated Git hooks.
Git LFS initialized.
Cloning into '/content/model_weights/faster-distil-whisper-large-v3'...
remote: Enumerating objects: 13, done.
remote: Total 13 (delta 0), reused 0 (delta 0), pack-reused 13 (from 1)
Receiving objects: 100% (13/13), 884.44 KiB | 14.50 MiB/s, done.
Resolving deltas: 100% (1/1), done.
  model.bin: 1.51 GB ✓


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

  model-00001-of-00004.safetensors: 4.87 GB ✓
  model-00004-of-00004.safetensors: 2.06 GB ✓

  imagebind_huge.pth: 4.80 GB ✓

=== Model verification summary ===
  ✓ Whisper
  ✓ ImageBind
  ✓ MiniCPM shard1

✓ All models ready. Safe to continue.


In [13]:
#symlink model weights to where videorag expects it
import os

# VideoRAG's code uses hardcoded relative paths for model weights.
# Symlinks make our /content/model_weights/ files appear where the code looks,
# without copying 22GB into the repo.

algo_dir = '/content/VideoRAG/VideoRAG-algorithm'

links = [
    (disk_paths['whisper'], f'{algo_dir}/faster-distil-whisper-large-v3'),
    (disk_paths['minicpm'], f'{algo_dir}/MiniCPM-V-2_6-int4'),
]

for src, dst in links:
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"✓ Symlink created: {os.path.basename(dst)} → {src}")
    else:
        print(f"✓ Symlink already exists: {os.path.basename(dst)}")

# ImageBind specifically expects a .checkpoints/ folder inside the repo
os.makedirs(f'{algo_dir}/.checkpoints', exist_ok=True)
imagebind_link = f'{algo_dir}/.checkpoints/imagebind_huge.pth'
if not os.path.exists(imagebind_link):
    os.symlink(disk_paths['imagebind'], imagebind_link)
    print(f"✓ Symlink created: imagebind_huge.pth")
else:
    print(f"✓ Symlink already exists: imagebind_huge.pth")

✓ Symlink created: faster-distil-whisper-large-v3 → /content/model_weights/faster-distil-whisper-large-v3
✓ Symlink created: MiniCPM-V-2_6-int4 → /content/model_weights/MiniCPM-V-2_6
✓ Symlink created: imagebind_huge.pth


In [14]:
%cd /content/VideoRAG/VideoRAG-algorithm
import os; print("CWD:", os.getcwd())

/content/VideoRAG/VideoRAG-algorithm
CWD: /content/VideoRAG/VideoRAG-algorithm


In [15]:
# ── LOAD VIDEOS FROM DRIVE ────────────────────────────────────────────────
# Assumes the one-time download cell has already populated Drive.
# This cell only reads what's there — no network, instant.

import glob, os

video_paths = sorted(glob.glob(f"{drive_paths['input_videos']}/*.mp4"))

if not video_paths:
    raise RuntimeError(
        f"No videos found in {drive_paths['input_videos']} — "
        f"run the one-time download cell first."
    )

print(f"=== {len(video_paths)} videos loaded from Drive ===")
for p in video_paths:
    print(f"  {os.path.basename(p)}: {os.path.getsize(p)/1e6:.0f}MB")

=== 6 videos loaded from Drive ===
  surg_00_4k_uncinectomy_and_middle_meatal_antrost.mp4: 81MB
  surg_01_4k_anterior_ethmoidectomy_prof_simon_car.mp4: 96MB
  surg_02_4k_frontal_recess_dissection_intact_bull.mp4: 113MB
  surg_03_replacement.mp4: 152MB
  surg_04_replacement.mp4: 115MB
  surg_05_basic_fess_step_by_step.mp4: 46MB


In [16]:
#deepseek setup and videorag config, colab is already running a loop so use nest_asyncio to run a nested loop.
import nest_asyncio
nest_asyncio.apply()                      # (1) Failure 9: Colab already runs an event loop
from google.colab import userdata

import os, sys, httpx, asyncio, types, torch
sys.path.insert(0, '/content/VideoRAG/VideoRAG-algorithm')

# (2) CUDA-COLD GUARD — the most important assertion in the notebook
if torch.cuda.is_initialized():
    raise RuntimeError(
        "CUDA already initialized — restart runtime. Something before this "
        "cell called torch.cuda.*. Indexing forks a subprocess and will crash."
    )
print("✓ CUDA not initialized — safe for multiprocessing fork")

# (3) pytorchvideo shim (Failure 4) — re-applied because Cell 5c's shim
#     lives in module memory; re-asserting here is cheap insurance
import torchvision.transforms.functional as F
fake = types.ModuleType('torchvision.transforms.functional_tensor')
for fn in ['rgb_to_grayscale','adjust_brightness','adjust_contrast',
           'adjust_saturation','adjust_hue']:
    if hasattr(F, fn): setattr(fake, fn, getattr(F, fn))
sys.modules['torchvision.transforms.functional_tensor'] = fake

# (4) clear stale videorag modules (Failure 10: module caching)
for mod in [k for k in sys.modules if 'videorag' in k]:
    del sys.modules[mod]

# (5) DeepSeek key from Colab secrets (not hardcoded!)
os.environ["DEEPSEEK_API_KEY"] = userdata.get('DS_TOKEN')
key = os.environ["DEEPSEEK_API_KEY"]
if not key.startswith("sk-"):
    raise ValueError("✗ DEEPSEEK_API_KEY looks wrong")
print(f"✓ DEEPSEEK_API_KEY: {key[:6]}...{key[-4:]}")

# (6) live test call — fail here, not 20 minutes into indexing
async def test_deepseek():
    async with httpx.AsyncClient() as c:
        r = await c.post("https://api.deepseek.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json={"model": "deepseek-chat",
                  "messages": [{"role": "user", "content": "say ok"}],
                  "max_tokens": 5})
        print("✓ DeepSeek reachable" if r.status_code == 200
              else f"✗ DeepSeek error {r.status_code}: {r.text[:200]}")
await test_deepseek()

✓ CUDA not initialized — safe for multiprocessing fork
✓ DEEPSEEK_API_KEY: sk-758...21f6
✓ DeepSeek reachable


In [17]:
# ── Cell 9.5 — pre-flight before indexing ─────────────────────────────────
import os, torch

# (1) Indexing runs on LOCAL disk (fast, no FUSE stalls during per-clip ASR).
#     The finished index gets copied to Drive at the end of Cell 10.
workdir       = f"/content/workdir_{run_id}"      # LOCAL — hot I/O lives here
drive_workdir = drive_paths['workdir']            # Drive — final destination
analysis_dir  = drive_paths['inspection']

os.makedirs(workdir, exist_ok=True)
os.makedirs(drive_workdir, exist_ok=True)
os.makedirs(analysis_dir, exist_ok=True)

# (2) Final CUDA-cold guard
if torch.cuda.is_initialized():
    raise RuntimeError("CUDA is HOT — restart runtime; insert_video() forks.")
print("✓ CUDA still cold — safe to fork")

# (3) Confirm the run
print(f"\nRun ID:        {run_id}  ({run_note})")
print(f"Local workdir: {workdir}   (indexing scratch + index)")
print(f"Drive workdir: {drive_workdir}   (final copy destination)")
existing = os.listdir(workdir)
if existing:
    print(f"⚠ local workdir NOT empty ({len(existing)} items) — left over from the hung run.")
    print("   Clear it for a clean run (see note below).")
else:
    print("✓ local workdir empty — clean run")

print(f"\nVideos to index ({len(video_paths)}):")
for p in video_paths:
    print(f"  {os.path.basename(p)}")

✓ CUDA still cold — safe to fork

Run ID:        001  (baseline)
Local workdir: /content/workdir_001   (indexing scratch + index)
Drive workdir: /content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir   (final copy destination)
✓ local workdir empty — clean run

Videos to index (6):
  surg_00_4k_uncinectomy_and_middle_meatal_antrost.mp4
  surg_01_4k_anterior_ethmoidectomy_prof_simon_car.mp4
  surg_02_4k_frontal_recess_dissection_intact_bull.mp4
  surg_03_replacement.mp4
  surg_04_replacement.mp4
  surg_05_basic_fess_step_by_step.mp4


In [18]:
import shutil, os
# remove the partial index from the hung Drive run
if os.path.exists(drive_paths['workdir']):
    shutil.rmtree(drive_paths['workdir'])
    print("✓ cleared partial Drive workdir")
# and any local leftover
if os.path.exists(f"/content/workdir_{run_id}"):
    shutil.rmtree(f"/content/workdir_{run_id}")
    print("✓ cleared local workdir")

✓ cleared partial Drive workdir
✓ cleared local workdir


In [19]:
# ── Cell 10 — indexing on local disk, then persist to Drive ───────────────
import time, shutil
from videorag._llm import deepseek_bge_config
from videorag import VideoRAG

start = time.time()
print(f"=== Indexing → LOCAL workdir: {workdir} ===\n")

videorag = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir,          # LOCAL (set in 9.5)
)

videorag.insert_video(video_path_list=video_paths)

print(f"\n✓ Indexing complete in {(time.time()-start)/60:.1f} min")

# persist the finished index to Drive in ONE bulk copy (FUSE handles this fine)
print(f"Copying index → Drive: {drive_workdir}")
shutil.copytree(workdir, drive_workdir, dirs_exist_ok=True)
print(f"✓ Index on Drive: {drive_workdir}")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  warnings.warn(

  warnings.warn(

  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)



=== Indexing → LOCAL workdir: /content/workdir_001 ===



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Spliting Video surg_00_4k_uncinectomy_and_middle_meatal_antrost: 100%|██████████| 9/9 [00:09<00:00,  1.01s/it]
Speech Recognition surg_00_4k_uncinectomy_and_middle_meatal_antrost: 100%|██████████| 9/9 [01:27<00:00,  9.70s/it]
Saving Video Segments surg_00_4k_uncinectomy_and_middle_meatal_antrost: 100%|██████████| 9/9 [02:28<00:00, 16.48s/it]
Captioning Video surg_00_4k_uncinectomy_and_middle_meatal_antrost:   0%|          | 0/9 [00:00<?, ?it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(

Captioning Video surg_00_4k_uncinectomy_and_middle_meatal_antrost: 100%|██████████| 9/9 [01:33<00:00, 10.34s/it]
Encoding Video Segments surg_00_4k_uncinectomy_and_middle_meatal_antrost: 100%|██████████| 5/5 [00:32<00:00,  6.59s/it]
Spliting Vi

Loading bge-m3 onto GPU...


...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

✓ bge-m3 loaded


✓ Indexing complete in 57.6 min
Copying index → Drive: /content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir
✓ Index on Drive: /content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir


Post-index analysis

In [20]:
%cd /content/VideoRAG/VideoRAG-algorithm
import glob, json, networkx as nx

wd = "/content/workdir_001"

# 1. The graph — are entities surgical & specific, or generic mush?
g = glob.glob(f"{wd}/*.graphml")
if g:
    G = nx.read_graphml(g[0])
    print(f"=== GRAPH: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges ===\n")
    for n, d in list(G.nodes(data=True))[:30]:
        print(f"  [{d.get('entity_type','?')}] {n}")

# 2. A transcript — how badly did Whisper mangle anatomy/instruments?
t = glob.glob(f"{wd}/*transcript*.json") or glob.glob(f"{wd}/**/*transcript*.json", recursive=True)
print("\n=== TRANSCRIPT FILES ===", t)

# 3. A caption — generic ("a surgical procedure") or specific (names instruments)?
c = glob.glob(f"{wd}/*caption*.json") or glob.glob(f"{wd}/**/*caption*.json", recursive=True)
print("=== CAPTION FILES ===", c)

/content/VideoRAG/VideoRAG-algorithm
=== GRAPH: 375 nodes, 460 edges ===

  ["EVENT"] "UNSONECTOMY"
  ["ORGANIZATION"] "BACKBITER"
  ["GEO"] "UNCINATE PROCESS"
  ["GEO"] "MIDDLE TURBINATE"
  ["ORGANIZATION"] "DOUBLE RIGHT-ANGLED BALL PROBE"
  ["GEO"] "FRONTAL PROCESS OF THE MAXILLA"
  ["ORGANIZATION"] "SICKLE KNIFE"
  ["ORGANIZATION"] "BACKBITING FORCEPS"
  ["PERSON"] "SURGEON"
  ["GEO"] "UNSINATE PROCESS"
  ["GEO"] "NASAL MUCOSA"
  ["GEO"] "TURBINATE BONES"
  ["ORGANIZATION"] "BLAKSTEY-WELLS FORCEPS"
  ["ORGANIZATION"] "KEROSEN PUNCH"
  ["ORGANIZATION"] "MICRODEBRIDER"
  ["ORGANIZATION"] "DOUBLE RIGHT ANGLE BOARD PROBE"
  ["ORGANIZATION"] "BLEXUEL FORCEPS"
  ["GEO"] "UNSNITNET PROCESS"
  ["GEO"] "UNSIGNATE PROCESS"
  ["GEO"] "UNSNUCK PROCESS"
  ["GEO"] "LATERAL NASAL WALL"
  ["GEO"] "MAXILLARY SINUS"
  ["GEO"] "NATURAL OSTEUM"
  ["GEO"] "EAR CANAL"
  ["PERSON"] "PATIENT"
  ["ORGANIZATION"] "THROUGH-BITING FORCEPS"
  ["ORGANIZATION"] "DOUBLE RIGHT ANGLE BALL PROBE"
  ["EVENT"] "MEDICAL

In [21]:
import json
wd = "/content/workdir_001"

# transcript — confirm ASR damage scale
t = json.load(open(f"{wd}/surg_00_4k_uncinectomy_and_middle_meatal_antrost_transcripts.json"))
print("=== TRANSCRIPT (surg_00) ===")
for k in list(t.keys())[:4]:
    print(f"[{k}] {t[k][:200]}\n")

# caption — generic or specific?
c = json.load(open(f"{wd}/surg_00_4k_uncinectomy_and_middle_meatal_antrost_captions.json"))
print("=== CAPTIONS (surg_00) ===")
for k in list(c.keys())[:4]:
    print(f"[{k}] {c[k][:250]}\n")

=== TRANSCRIPT (surg_00) ===
[0] [0.00s -> 22.16s]  The first step of the unsonectomy involves inserting the small backbiter between the middle
[22.16s -> 27.00s]  turbinate and the free edge of the unsinate process.
[27.00s -> 29.98

[1] [0.00s -> 6.72s]  this narrow cleft in the vertical position followed by rotating it laterally you
[6.72s -> 12.72s]  can see here it's taking several bites to come forward along the whole length of t

[2] [0.00s -> 4.84s]  vertical. As you can see in this case, it's angling slightly towards the middle
[4.84s -> 13.38s]  turbine, so therefore the angle of this superior cut is often not at 90 degrees. Th

[3] [0.00s -> 5.04s]  And by pulling it forward, we've fractured that unsnit process into clear view.
[6.64s -> 13.84s]  Using through-biting 45-degree Blakstey-Wells forceps, we're now able to amputate t

=== CAPTIONS (surg_00) ===
[0] The video begins with a close-up view of the nasal cavity, focusing on the surgical process. The camera captures a d

In [23]:
%cd /content/VideoRAG/VideoRAG-algorithm
import os
wd = "/content/workdir_001"
out = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs"

# if the analysis script exists on surgery branch:
if os.path.exists("scripts/extract_indexing_analysis.py"):
    !python scripts/extract_indexing_analysis.py --workdir {wd} --output-dir {out}
else:
    print("no analysis script — saving raw artifacts instead")

/content/VideoRAG/VideoRAG-algorithm
Reading workdir: /content/workdir_001
Loaded graph: 375 nodes, 460 edges
Writing analysis to: /content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs
Wrote 8 analysis files: corpus_map, clips, chunks, entities, relationships, graph_summary, provenance_check, aggregate_stats


In [24]:
import os
wd_drive = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir"
for f in sorted(os.listdir(wd_drive)):
    print(" ", f)

  _cache
  graph_chunk_entity_relation.graphml
  kv_store_llm_response_cache.json
  kv_store_text_chunks.json
  kv_store_video_path.json
  kv_store_video_segments.json
  surg_00_4k_uncinectomy_and_middle_meatal_antrost_captions.json
  surg_00_4k_uncinectomy_and_middle_meatal_antrost_transcripts.json
  surg_01_4k_anterior_ethmoidectomy_prof_simon_car_captions.json
  surg_01_4k_anterior_ethmoidectomy_prof_simon_car_transcripts.json
  surg_02_4k_frontal_recess_dissection_intact_bull_captions.json
  surg_02_4k_frontal_recess_dissection_intact_bull_transcripts.json
  surg_03_replacement_captions.json
  surg_03_replacement_transcripts.json
  surg_04_replacement_captions.json
  surg_04_replacement_transcripts.json
  surg_05_basic_fess_step_by_step_captions.json
  surg_05_basic_fess_step_by_step_transcripts.json
  vdb_chunks.json
  vdb_entities.json
  vdb_video_segment_feature.json


Setup to retrieve

In [ ]:
# ── Retrieval — load saved index from Drive, run a query ──────────────────
# PREREQ: engine setup cells already run this session (deps, weights,
# symlinks, DeepSeek key). This does NOT re-index — it loads workdir.
import os, sys, multiprocessing
sys.path.insert(0, '/content/VideoRAG/VideoRAG-algorithm')

# Keys don't persist across sessions — set again
from google.colab import userdata
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DS_TOKEN")
os.environ["HF_TOKEN"]         = userdata.get("HF_TOKEN")

# Retrieval uses spawn (not the indexing cold-fork approach) — its own working config
multiprocessing.set_start_method('spawn', force=True)

from videorag._llm import deepseek_bge_config
from videorag import VideoRAG, QueryParam

workdir = drive_paths['workdir']   # SAME workdir written by indexing → loads the index

print(f"Loading index from: {workdir}")
videorag = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir,            # no insert_video() = no re-indexing
)
print("✓ Index loaded (graph + embeddings from Drive)")

# ── Run a query ───────────────────────────────────────────────────────────
query = "What instruments are used during the uncinectomy?"
param = QueryParam(mode="videorag")   # confirm the mode string your branch uses

print(f"\nQuery: {query}\n")
answer = videorag.query(query, param=param)
print("=== Answer ===")
print(answer)

In [28]:
import time
from videorag import VideoRAG, QueryParam
from videorag._llm import deepseek_bge_config
import multiprocessing

multiprocessing.set_start_method('spawn', force=True)   # retrieval re-captioning fork safety

drive_workdir = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir"
analysis_dir  = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs"

videorag = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=drive_workdir,
    analysis_output_dir=analysis_dir,   # turns on QueryRecorder
)

queries = [
    ("q01_uncinectomy_instruments",
     "What instruments are used during the uncinectomy?"),
    ("q02_uncinectomy_steps",
     "What are the steps of performing an uncinectomy?"),
    ("q03_crossvideo_landmarks",
     "What anatomical landmarks are referenced across the different sinus procedures?"),
]

videorag.load_caption_model()   # loads MiniCPM-V for query-time re-captioning

for qid, query in queries:
    print(f"\n{'='*70}\nQUERY {qid}\n{'='*70}\n{query}\n")
    start = time.time()
    param = QueryParam(mode="videorag", query_id=qid)
    param.wo_reference = False
    response = videorag.query(query=query, param=param)
    print(f"\n--- Response ({time.time()-start:.0f}s) ---\n{response}")
    # final answer to a flat file too (matches your old pattern)
    with open(f"{analysis_dir}/query_{qid}_response.txt", "w") as f:
        f.write(f"Query: {query}\n\nResponse:\n{response}")

print(f"\n✓ Answers + intermediates in {analysis_dir}/queries/<query_id>/")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


QUERY q01_uncinectomy_instruments
What instruments are used during the uncinectomy?

The instruments used during the uncinectomy.
Retrieved Text Segments {'surg_00_4k_uncinectomy_and_middle_meatal_antrost_0', 'surg_05_basic_fess_step_by_step_10', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_2', 'surg_05_basic_fess_step_by_step_11', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_1', 'surg_05_basic_fess_step_by_step_9'}
The uncinectomy procedure involving the use of specific instruments.
Retrieved Visual Segments {'surg_02_4k_frontal_recess_dissection_intact_bull_0', 'surg_03_replacement_2', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_0', 'surg_05_basic_fess_step_by_step_1'}
9 Video Segments remain after filtering
Remain segments ['surg_00_4k_uncinectomy_and_middle_meatal_antrost_0', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_1', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_2', 'surg_02_4k_frontal_recess_dissection_intact_bull_0', 'surg_03_replacement_2', 'surg

Captioning Segments for Given Query: 100%|██████████| 9/9 [03:03<00:00, 20.38s/it]



--- Response (209s) ---
Based on the retrieved information, the uncinectomy procedure utilizes several specialized instruments, each for a specific step in the process. The procedure begins with the initial incision and progresses through the removal of bone and tissue.

The key instruments used during an uncinectomy include:

*   **Backbiter (Backbiting Forceps):** This is the primary instrument for the first step. It is inserted between the middle turbinate and the free edge of the uncinate process and twisted to make the initial cut. It is also used to make the superior cut, as it is considered much safer than a sickle knife for this purpose due to the angle of the uncinate process [1, 2].
*   **Sickle Knife:** While mentioned as an alternative for the superior cut, it is noted that using a backbiting forceps is safer [2].
*   **Double Right-Angled Ball Probe:** This instrument is used to fracture the uncinate process forward, detaching it from the lateral nasal wall. The tip of th

Captioning Segments for Given Query: 100%|██████████| 7/7 [02:31<00:00, 21.70s/it]



--- Response (181s) ---
# Steps of Performing an Uncinectomy

An uncinectomy is a critical first step in functional endoscopic sinus surgery (FESS), involving the removal of the uncinate process to access the sinuses. Based on the retrieved surgical videos and transcripts, the procedure follows a systematic sequence of steps.

## 1. Initial Exposure and Positioning

The procedure begins by inserting a small backbiter between the middle turbinate and the free edge of the uncinate process. The surgeon must twist the backbiter to navigate it into this narrow cleft [1]. This initial positioning is crucial for accessing the vertical portion of the uncinate process.

## 2. Sequential Biting Along the Uncinate Process

Once positioned, the surgeon takes several bites to advance along the entire length of the uncinate process. The backbiting forceps are rotated laterally to work through the vertical cleft, gradually exposing more of the anatomy [2]. A second cut is made superiorly, and while 

Captioning Segments for Given Query: 100%|██████████| 9/9 [03:04<00:00, 20.47s/it]



--- Response (212s) ---
Based on the provided information, several key anatomical landmarks are consistently referenced across different sinus procedures, particularly during Functional Endoscopic Sinus Surgery (FESS). These landmarks are crucial for safe and effective navigation during surgery.

### Key Anatomical Landmarks

The following landmarks are frequently mentioned across multiple procedures:

*   **Middle Turbinate:** This structure is a central reference point. It is often used to locate other structures and is a key landmark during procedures like maxillary antrostomy and anterior ethmoidectomy. The basal lamella, the horizontal attachment of the middle turbinate, is highlighted as the dividing line between the anterior and posterior ethmoid air cells [1].
*   **Uncinate Process:** This is a key landmark for accessing the maxillary sinus. It is often described as resembling a half moon and is the first structure addressed in a maxillary antrostomy [1]. The process is refle

In [29]:
import os
qdir = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/queries"
if os.path.exists(qdir):
    for q in sorted(os.listdir(qdir)):
        print(q, "→", os.listdir(f"{qdir}/{q}"))
else:
    print("no queries/ folder — recorder didn't fire")

q01_uncinectomy_instruments → ['query.json', 'path3.json', 'path1.json', 'path2.json', 'filter.json', 'reformulations.json', 'recaption.json', 'generation.json', 'timing.json']
q02_uncinectomy_steps → ['query.json', 'path3.json', 'path1.json', 'path2.json', 'filter.json', 'reformulations.json', 'recaption.json', 'generation.json', 'timing.json']
q03_crossvideo_landmarks → ['query.json', 'path3.json', 'path1.json', 'path2.json', 'filter.json', 'reformulations.json', 'recaption.json', 'generation.json', 'timing.json']


In [30]:
import time, multiprocessing
from videorag import VideoRAG, QueryParam
from videorag._llm import deepseek_bge_config

multiprocessing.set_start_method('spawn', force=True)

drive_workdir = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/workdir"
# dedicated subfolder for this batch → clean separation
analysis_dir  = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/opnote_readiness"

import os
os.makedirs(analysis_dir, exist_ok=True)

videorag = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=drive_workdir,
    analysis_output_dir=analysis_dir,
)
videorag.load_caption_model()

opnote_probe_queries = [
    ("op01_procedure",   "What surgical procedure is being performed in this video?"),
    ("op02_steps",       "List the sequence of steps performed during the procedure."),
    ("op03_instruments", "What instruments are used, and at what step is each used?"),
    ("op04_anatomy",     "What anatomical structures are identified and operated on?"),
    ("op05_findings",    "What anatomical findings or abnormalities are described?"),
]

for qid, query in opnote_probe_queries:
    print(f"\n{'='*70}\n{qid}\n{'='*70}\n{query}\n")
    start = time.time()
    param = QueryParam(mode="videorag", query_id=qid)
    param.wo_reference = False
    response = videorag.query(query=query, param=param)
    print(f"\n--- Response ({time.time()-start:.0f}s) ---\n{response}")
    with open(f"{analysis_dir}/{qid}_response.txt", "w") as f:
        f.write(f"Query: {query}\n\nResponse:\n{response}")

print(f"\n✓ → {analysis_dir}/queries/<id>/")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


op01_procedure
What surgical procedure is being performed in this video?

The surgical procedure being performed in this video.
Retrieved Text Segments {'surg_02_4k_frontal_recess_dissection_intact_bull_2', 'surg_05_basic_fess_step_by_step_14', 'surg_02_4k_frontal_recess_dissection_intact_bull_1', 'surg_02_4k_frontal_recess_dissection_intact_bull_0', 'surg_05_basic_fess_step_by_step_12', 'surg_05_basic_fess_step_by_step_13'}
A surgical procedure is being performed in the video.
Retrieved Visual Segments {'surg_02_4k_frontal_recess_dissection_intact_bull_0', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_7', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_0', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_8'}
9 Video Segments remain after filtering
Remain segments ['surg_00_4k_uncinectomy_and_middle_meatal_antrost_0', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_7', 'surg_00_4k_uncinectomy_and_middle_meatal_antrost_8', 'surg_02_4k_frontal_recess_dissection_intact_bull_0', 

Captioning Segments for Given Query: 100%|██████████| 9/9 [03:01<00:00, 20.21s/it]



--- Response (207s) ---
Based on the retrieved information, the video demonstrates an **uncinectomy and middle meatal antrostomy**, which are key steps in endoscopic sinus surgery (ESS), specifically functional endoscopic sinus surgery (FESS). The procedure focuses on removing the uncinate process to access and enlarge the natural drainage pathway of the maxillary sinus.

The surgery begins with the insertion of a backbiter instrument between the middle turbinate and the free edge of the uncinate process [1]. After removing the uncinate process, the surgeon works to create a "nice opening into the maxillary sinus" while carefully preserving the mucosa around the natural ostium [2]. The goal is to "stretch the opening of the natural and auxiliary sinus ostium to oppose those mucosal edges and make sure that the common drainage pathway is still intact" [3]. A middle meatal spacer made of hemostatic material is then placed to prevent scarring between the middle turbinate and the lateral 

Captioning Segments for Given Query: 100%|██████████| 8/8 [02:42<00:00, 20.28s/it]


...


--- Response (220s) ---
Based on the retrieved information, here is a summary of the surgical instruments used during a Functional Endoscopic Sinus Surgery (FESS), along with the specific step or anatomical target for each.

### Key Instruments and Their Uses in FESS

The procedures involve a sequence of steps, each requiring specific tools for dissection, removal, and visualization. The primary goal is to open sinus air cells, improve drainage, and preserve healthy mucosa.

#### 1. Uncinectomy (Removal of the Uncinate Process)

This is an early step to gain access to the maxillary sinus and ethmoid bulla.

- **Backbiter (Backbiting Forceps):** Used to make the initial cut in the uncinate process. It is inserted between the middle turbinate and the free edge of the uncinate process, often requiring a twist to get into position [1, 2].
- **Double Right-Angled Ball Probe:** Used to fracture the uncinate process away from the lateral nasal wall, often after a backbiter has been used [3].

Captioning Segments for Given Query: 100%|██████████| 8/8 [02:59<00:00, 22.43s/it]



--- Response (208s) ---
Based on the retrieved information, a Functional Endoscopic Sinus Surgery (FESS) involves the identification and surgical manipulation of several key anatomical structures within the nasal cavity and paranasal sinuses. The procedures described are systematic, moving through specific "gates" or landmarks to ensure safe and effective treatment.

### Key Anatomical Structures Identified and Operated On

The surgery focuses on opening the four main sinus cavities and their associated drainage pathways. The following structures are explicitly identified and manipulated during the procedure:

- **Turbinates:** The **middle turbinate** and **superior turbinate** are key landmarks. The middle turbinate's lateral attachment, the **basal (or ground) lamella**, is a critical boundary that divides the anterior from the posterior ethmoid sinuses and must be carefully entered to access the posterior cells [3, 5].
- **Uncinate Process:** This is referred to as "gate number tw

Captioning Segments for Given Query: 100%|██████████| 9/9 [03:02<00:00, 20.29s/it]



--- Response (211s) ---
Based on the retrieved information, several anatomical findings, landmarks, and procedural observations are described across the surgical videos. These findings are crucial for guiding endoscopic sinus surgery (FESS) and ensuring patient safety.

### Key Anatomical Findings and Landmarks

**1. The Uncinate Process and Surrounding Structures**
The uncinate process is a key landmark in the initial steps of FESS. It is described as a curved structure resembling a "half moon" [1]. The procedure involves fracturing and amputating this process to access the maxillary sinus ostium [2]. The "blind sac" of the uncinate process, which is covered by mucosa, is also a notable finding after the bone has been removed [3, 4].

**2. The Maxillary Sinus and Antrostomy Site**
After removing the inferior portion of the uncinate process, the natural ostium of the maxillary sinus becomes visible [2]. The ideal shape of a completed maxillary antrostomy is described as "pear-shaped,"

In [31]:
import networkx as nx, json, collections, glob, os, re

wd = "/content/workdir_001"
# one clean baseline-analysis folder for run 1
base = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/baseline_analysis"
os.makedirs(base, exist_ok=True)

G = nx.read_graphml(f"{wd}/graph_chunk_entity_relation.graphml")

# 1. nodes + edges + type counts
nodes = [{"name": n, **d} for n, d in G.nodes(data=True)]
types = collections.Counter(d.get("entity_type","?") for _,d in G.nodes(data=True))
json.dump(nodes, open(f"{base}/all_nodes.json","w"), indent=2, default=str)
json.dump([{"src":u,"tgt":v,**d} for u,v,d in G.edges(data=True)],
          open(f"{base}/all_edges.json","w"), indent=2, default=str)
json.dump(dict(types), open(f"{base}/type_counts.json","w"), indent=2)

# 2. fragmentation
def norm(s): return re.sub(r'[^a-z]','', s.lower())
groups = collections.defaultdict(list)
for n in G.nodes(): groups[norm(n)[:6]].append(n)
frag = {k:v for k,v in groups.items() if len(v)>1}
json.dump(frag, open(f"{base}/fragmentation.json","w"), indent=2)

# 3. transcripts (full, for ASR error cataloguing)
asr = {os.path.basename(tf): json.load(open(tf)) for tf in glob.glob(f"{wd}/*_transcripts.json")}
json.dump(asr, open(f"{base}/all_transcripts.json","w"), indent=2)

# 4. captions (for genericness review)
caps = {os.path.basename(cf): json.load(open(cf)) for cf in glob.glob(f"{wd}/*_captions.json")}
json.dump(caps, open(f"{base}/all_captions.json","w"), indent=2)

print(f"✓ saved to {base}")
print("  type distribution:", dict(types))
print(f"  {len(nodes)} nodes, {G.number_of_edges()} edges, "
      f"{len(frag)} fragmentation groups")
for f in sorted(os.listdir(base)):
    print("   ", f)

✓ saved to /content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/baseline_analysis
  type distribution: {'"EVENT"': 55, '"ORGANIZATION"': 95, '"GEO"': 197, '"PERSON"': 28}
  375 nodes, 460 edges, 47 fragmentation groups
    all_captions.json
    all_edges.json
    all_nodes.json
    all_transcripts.json
    fragmentation.json
    type_counts.json


In [32]:
import json
qdir = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/opnote_readiness/queries"
import os
# pick one query's generation dump and see its structure
g = json.load(open(f"{qdir}/op02_steps/generation.json"))
print("keys:", list(g.keys()) if isinstance(g, dict) else type(g))
print(json.dumps(g, indent=2, default=str)[:1500])

keys: ['video_data_csv', 'chunk_data_text', 'full_system_prompt', 'response', 'token_counts']
{
  "video_data_csv": "\"video_name\",\t\"start_time\",\t\"end_time\",\t\"content\"\n\"surg_02_4k_frontal_recess_dissection_intact_bull\",\t\"0:0:0\",\t\"0:0:30\",\t\"Caption:\nThe video begins with a gradient blue background and the word \"OLYMPUS\" in white, followed by a tagline \"Your Vision, Our Future.\" It transitions to display the logo for \"Olympus Academy,\" which includes a stylized 'O' and the text \"Olympus Academy\" in blue. The scene shifts to a plain blue screen with the title \"Frontal Recess Dissection: Intact Bulla Technique,\" indicating an educational or instructional context.Subsequently, the video introduces Prof A Simon Carney, identified as an ENT Surgeon affiliated with Flinders University in Adelaide, South Australia. This suggests that Prof Carney is likely providing expert commentary or instruction on the medical procedure being demonstrated.The focus then moves t

In [33]:
import json, os
from openai import OpenAI

qdir = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/opnote_readiness/queries"
out  = "/content/surgical_drive/MyDrive/videorag_surgical/runs/001/inspection_outputs/opnote_readiness"

contexts = []
for qid in ["op01_procedure","op02_steps","op03_instruments","op04_anatomy","op05_findings"]:
    g = json.load(open(f"{qdir}/{qid}/generation.json"))
    contexts.append(
        f"### Retrieved for '{qid}' ###\n"
        f"[VISUAL CAPTIONS]\n{g.get('video_data_csv','')}\n\n"
        f"[TEXT CHUNKS]\n{g.get('chunk_data_text','')}"
    )
full_context = "\n\n".join(contexts)

template = """You are an ENT surgeon writing an operative note from intraoperative video documentation.
Using ONLY the provided context, complete the operative note below.
For any section the context does not support, write "NOT DOCUMENTED IN SOURCE" — do not invent clinical detail.

OPERATIVE NOTE
==============
PROCEDURE PERFORMED:
INDICATION:
ANATOMY ENCOUNTERED:
INSTRUMENTS USED:
OPERATIVE STEPS:
FINDINGS:
COMPLICATIONS:
==============

CONTEXT:
""" + full_context

client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")
resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role":"user","content":template}],
    max_tokens=2000,
)
note = resp.choices[0].message.content
print(note)

with open(f"{out}/opnote_dress_rehearsal.txt","w") as f:
    f.write("=== OP-NOTE GENERATION PREVIEW (isolated) ===\n\n")
    f.write("Context fed: video_data_csv + chunk_data_text from 5 op-note queries\n\n")
    f.write(note)
# also save the exact context fed, for reproducibility / Phase B comparison
with open(f"{out}/opnote_dress_rehearsal_context.txt","w") as f:
    f.write(full_context)
print(f"\n✓ saved note + context to {out}/")

OPERATIVE NOTE
PROCEDURE PERFORMED: Functional endoscopic sinus surgery (FESS) including uncinectomy, middle meatal antrostomy, anterior ethmoidectomy, posterior ethmoidectomy, sphenoidotomy, and frontal sinusotomy.

INDICATION: NOT DOCUMENTED IN SOURCE

ANATOMY ENCOUNTERED: Inferior turbinate, middle turbinate, septum, left septal spur, maxillary sinus, ethmoid bulla, ethmoid air cells, ground lamella, lamina papyracea, skull base, posterior ethmoidal artery, frontal recess, sphenoid sinus, natural maxillary sinus ostium, agger nasi cell, frontal sinus.

INSTRUMENTS USED: Backbiter, through-biting 45-degree Blakesley-Wells forceps, double right angle ball probe, kerrison punch (2mm, 130° upward cutting), microdebrider (Olympus Diego Elite, Type A blade), frontal sinus seeker, Housman's frontal 70-degree punch, upward turning frontal punch, olive tip suction, kerosene/mushroom punch, through-cutting instruments, microdebrider.

OPERATIVE STEPS:
- Uncinectomy: small backbiter inserted b